## 04 — RAG question answering
Embeds the question with the same model as the chunks, retrieves the top-k chunks
from the AI Search index, builds a grounded prompt and asks the LLM.

In [0]:
%pip install databricks-ai-search openai

In [0]:
dbutils.library.restartPython()

In [0]:
for key in ["catalog", "silver_schema", "gold_schema", "rag_schema"]:
    dbutils.widgets.text(key, "")

In [0]:
%run ./rag_functions

In [0]:
example = answer_question("Which actors from Avengers: Infinity War appeared on Jimmy Kimmel Live?")

logger.info(
    "Example answer (%s prompt tokens): %s | sources: %s",
    example["prompt_tokens"],
    example["answer"],
    [source["chunk_id"] for source in example["sources"]],
)

In [0]:
%sql
SELECT
  video_id,
  title,
  channel_title,
  category_name,
  views,
  length(description) AS desc_len,
  left(description, 300) AS description_start
FROM dbr_dev_ua5816bd.natalkamartinuk55_rag.rag_video_documents
WHERE length(description) BETWEEN 300 AND 1500
  AND views < 200000
  AND category_name IN ('Howto & Style', 'Science & Technology', 'Education', 'Travel & Events', 'Autos & Vehicles')
ORDER BY rand(42)
LIMIT 15;

In [0]:
filtered = retrieve("funny prank videos", filters={"category_name": "Comedy"})

logger.info(
    "Filtered retrieval (category_name = Comedy): %s",
    [(r["category_name"], r["title"], round(r["score"], 3)) for r in filtered],
)